In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/06 04:07:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_green = spark.read.parquet('../code/data/pq/green/*/*')

In [3]:
df_green = df_green.withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime')

In [7]:
df_green.createOrReplaceTempView('green')

In [64]:
df_green_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', pickup_datetime) AS hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_of_records

    -- Additional calculations

FROM green
WHERE pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY 1, 2
ORDER BY 1, 2
""")

In [65]:
df_green_revenue.show()

+-------------------+----+------------------+-----------------+
|               hour|zone|            amount|number_of_records|
+-------------------+----+------------------+-----------------+
|2020-01-01 00:00:00|   7| 769.7299999999998|               45|
|2020-01-01 00:00:00|  17|195.03000000000003|                9|
|2020-01-01 00:00:00|  18|               7.8|                1|
|2020-01-01 00:00:00|  22|              15.8|                1|
|2020-01-01 00:00:00|  24|              87.6|                3|
|2020-01-01 00:00:00|  25|             531.0|               26|
|2020-01-01 00:00:00|  29|              61.3|                1|
|2020-01-01 00:00:00|  32| 68.94999999999999|                2|
|2020-01-01 00:00:00|  33| 317.2700000000001|               11|
|2020-01-01 00:00:00|  35|129.96000000000004|                5|
|2020-01-01 00:00:00|  36|            295.34|               11|
|2020-01-01 00:00:00|  37|175.67000000000002|                6|
|2020-01-01 00:00:00|  38| 98.7899999999

In [66]:
df_green_revenue \
.repartition(20) \
.write.parquet('../code/data/report/revenue/green', mode='overwrite')

In [28]:
df_yellow = spark.read.parquet('../code/data/pq/yellow/*/*')

In [29]:
df_yellow = df_yellow.withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime')

In [30]:
df_yellow.createOrReplaceTempView('yellow')

In [67]:
df_yellow_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', pickup_datetime) AS hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_of_records

    -- Additional calculations

FROM yellow
WHERE pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY 1, 2
ORDER BY 1, 2
""")

In [69]:
df_yellow_revenue.show()

+-------------------+----+------------------+-----------------+
|               hour|zone|            amount|number_of_records|
+-------------------+----+------------------+-----------------+
|2020-01-01 00:00:00|   3|              25.0|                1|
|2020-01-01 00:00:00|   4|1004.3000000000002|               57|
|2020-01-01 00:00:00|   7| 455.1700000000001|               38|
|2020-01-01 00:00:00|  10|             42.41|                2|
|2020-01-01 00:00:00|  12|             107.0|                6|
|2020-01-01 00:00:00|  13|            1214.8|               56|
|2020-01-01 00:00:00|  14|               8.8|                1|
|2020-01-01 00:00:00|  15|             34.09|                1|
|2020-01-01 00:00:00|  17|220.20999999999998|                8|
|2020-01-01 00:00:00|  18|               5.8|                1|
|2020-01-01 00:00:00|  24| 754.9500000000002|               45|
|2020-01-01 00:00:00|  25|            324.35|               16|
|2020-01-01 00:00:00|  32|              

In [70]:
df_yellow_revenue \
.repartition(20) \
.write.parquet('../code/data/report/revenue/yellow', mode='overwrite')

25/03/05 20:23:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [74]:
df_green_revenue = spark.read.parquet('../code/data/report/revenue/green')

df_yellow_revenue = spark.read.parquet('../code/data/report/revenue/yellow')

In [75]:
df_green_revenue_temp = df_green_revenue \
    .withColumnsRenamed({'amount': 'green_amount', 'number_of_records': 'no_of_records_green'})

In [76]:
df_yellow_revenue_temp = df_yellow_revenue \
    .withColumnsRenamed({'amount': 'yellow_amount', 'number_of_records': 'no_of_records_yellow'})

In [77]:
df_join = df_green_revenue_temp.join(df_yellow_revenue_temp, on=['hour', 'zone'], how='outer')

In [78]:
df_join.show()

+-------------------+----+------------------+-------------------+------------------+--------------------+
|               hour|zone|      green_amount|no_of_records_green|     yellow_amount|no_of_records_yellow|
+-------------------+----+------------------+-------------------+------------------+--------------------+
|2020-01-01 00:00:00|  29|              61.3|                  1|              NULL|                NULL|
|2020-01-01 00:00:00|  33| 317.2700000000001|                 11|            255.56|                   8|
|2020-01-01 00:00:00|  62|             15.95|                  1|             61.43|                   1|
|2020-01-01 00:00:00|  71|              23.8|                  1|              NULL|                NULL|
|2020-01-01 00:00:00|  81|54.870000000000005|                  2|             30.32|                   1|
|2020-01-01 00:00:00| 125|              NULL|               NULL|           1342.07|                  68|
|2020-01-01 00:00:00| 126|              NULL| 

In [79]:
df_join.write.parquet('../code/data/report/revenue/total', mode='overwrite')

25/03/05 20:25:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [81]:
df_join = spark.read.parquet('../code/data/report/revenue/total')

In [82]:
df_join.show()

+-------------------+----+------------------+-------------------+------------------+--------------------+
|               hour|zone|      green_amount|no_of_records_green|     yellow_amount|no_of_records_yellow|
+-------------------+----+------------------+-------------------+------------------+--------------------+
|2020-01-01 00:00:00|   7| 769.7299999999998|                 45| 455.1700000000001|                  38|
|2020-01-01 00:00:00|  47|              13.3|                  1|               8.3|                   1|
|2020-01-01 00:00:00|  56|             99.69|                  3|              18.1|                   2|
|2020-01-01 00:00:00|  63|              51.9|                  2|              70.8|                   1|
|2020-01-01 00:00:00|  66|386.75000000000006|                 18|            260.55|                  10|
|2020-01-01 00:00:00|  75| 278.1400000000001|                 26| 958.3500000000001|                  69|
|2020-01-01 00:00:00|  77| 75.99000000000001| 

In [83]:
df_zones = spark.read.parquet('zones/')

In [84]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [86]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [95]:
df_result.drop('LocationID').show()

+-------------------+----+------------------+-------------------+------------------+--------------------+---------+--------------------+------------+
|               hour|zone|      green_amount|no_of_records_green|     yellow_amount|no_of_records_yellow|  Borough|                Zone|service_zone|
+-------------------+----+------------------+-------------------+------------------+--------------------+---------+--------------------+------------+
|2020-01-01 00:00:00|   7| 769.7299999999998|                 45| 455.1700000000001|                  38|   Queens|             Astoria|   Boro Zone|
|2020-01-01 00:00:00|  47|              13.3|                  1|               8.3|                   1|    Bronx|  Claremont/Bathgate|   Boro Zone|
|2020-01-01 00:00:00|  56|             99.69|                  3|              18.1|                   2|   Queens|              Corona|   Boro Zone|
|2020-01-01 00:00:00|  63|              51.9|                  2|              70.8|                

25/03/06 01:13:14 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 931706 ms exceeds timeout 120000 ms
25/03/06 01:13:14 WARN SparkContext: Killing executors is not supported by current scheduler.
25/03/06 01:13:18 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [ ]:
df_result.drop('LocationID').write.parquet('tmp/revenue')